### Bronze

Ingestão das 9 fontes sem correção de qualidade. Aplicados apenas os
ajustes necessários para leitura correta de cada formato (delimitador,
encoding, schema aninhado). O tratamento de qualidade é realizado
integralmente na camada Silver.

Cada tabela recebe 3 colunas de controle: timestamp de ingestão, arquivo
de origem e sistema/área de origem.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F

def add_audit_cols(df, source_file, source_system):
    return (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(source_file))
          .withColumn("_source_system", F.lit(source_system))
    )

#### Pedidos - cabeçalho

CSV delimitado por `;`. A coluna `payment_details` contém um JSON
serializado como string (aspas escapadas). Mantido como string nesta
camada; a extração ocorre na Silver.

In [ ]:
df_pedidos_cab_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", False)
    .csv(f"{RAW_PATH}/erp_pedidos_cabecalho_2025.csv")
)

df_pedidos_cab_bronze = add_audit_cols(df_pedidos_cab_raw, "erp_pedidos_cabecalho_2025.csv", "ERP")

(
    df_pedidos_cab_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.bronze_pedidos_cabecalho")
)

print(df_pedidos_cab_bronze.count(), "linhas")
display(df_pedidos_cab_bronze.limit(5))

#### Pedidos - itens

Delimitador `,`, diferente do cabeçalho (`;`) — ambos os arquivos são
originados do mesmo sistema, mas com exports distintos. `unit_price`
apresenta vírgula decimal entre aspas em parte dos registros.

In [ ]:
df_pedidos_itens_raw = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", False)
    .csv(f"{RAW_PATH}/erp_pedidos_itens_2025.csv")
)

df_pedidos_itens_bronze = add_audit_cols(df_pedidos_itens_raw, "erp_pedidos_itens_2025.csv", "ERP")

(
    df_pedidos_itens_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.bronze_pedidos_itens")
)

print(df_pedidos_itens_bronze.count(), "linhas")
display(df_pedidos_itens_bronze.limit(5))

#### Clientes (Excel)

Leitura via pandas e conversão para Spark DataFrame — abordagem mais
direta que configurar uma biblioteca de leitura de Excel para Spark dado
o volume reduzido do arquivo. Volumes do Unity Catalog são acessíveis
como path de sistema de arquivos padrão, sem o prefixo `/dbfs/`.

In [ ]:
import pandas as pd

pdf_clientes = pd.read_excel(f"{RAW_PATH}/crm_clientes_export.xlsx", dtype=str)
df_clientes_raw = spark.createDataFrame(pdf_clientes)
df_clientes_bronze = add_audit_cols(df_clientes_raw, "crm_clientes_export.xlsx", "CRM")

(
    df_clientes_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.bronze_clientes")
)

print(df_clientes_bronze.count(), "linhas")
display(df_clientes_bronze.limit(5))

#### Produtos (JSON aninhado)

Estrutura com `product`, `pricing` e `attributes` em níveis separados,
além de um array de tags. Mantido aninhado nesta camada; o flatten é
realizado na Silver, junto com a definição dos nomes de coluna finais.

In [ ]:
df_produtos_raw = spark.read.option("multiLine", True).json(f"{RAW_PATH}/cadastro_produtos_api_dump.json")
df_produtos_bronze = add_audit_cols(df_produtos_raw, "cadastro_produtos_api_dump.json", "Cadastro/API Produtos")

(
    df_produtos_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", True)
    .saveAsTable(f"{schema_name}.bronze_produtos")
)

print(df_produtos_bronze.count(), "linhas")
df_produtos_bronze.printSchema()

#### Canais (Excel)

In [ ]:
pdf_canais = pd.read_excel(f"{RAW_PATH}/comercial_canais.xlsx", dtype=str)
df_canais_raw = spark.createDataFrame(pdf_canais)
df_canais_bronze = add_audit_cols(df_canais_raw, "comercial_canais.xlsx", "Comercial")

(
    df_canais_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.bronze_canais")
)

display(df_canais_bronze)

#### Vendedores

In [ ]:
df_vendedores_raw = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(f"{RAW_PATH}/vendedores.csv")
)

df_vendedores_bronze = add_audit_cols(df_vendedores_raw, "vendedores.csv", "Comercial")

(
    df_vendedores_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.bronze_vendedores")
)

print(df_vendedores_bronze.count(), "linhas")
display(df_vendedores_bronze.limit(5))

#### Regiões

Arquivo legado, delimitado por pipe, com terminação de linha CRLF —
tratada automaticamente pelo leitor CSV do Spark.

In [ ]:
df_regioes_raw = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .option("inferSchema", False)
    .csv(f"{RAW_PATH}/legado_regioes_pipe.txt")
)

df_regioes_bronze = add_audit_cols(df_regioes_raw, "legado_regioes_pipe.txt", "Legado")

(
    df_regioes_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.bronze_regioes")
)

display(df_regioes_bronze)

#### Entregas (JSON aninhado)

In [ ]:
df_entregas_raw = spark.read.option("multiLine", True).json(f"{RAW_PATH}/logistica_entregas.json")
df_entregas_bronze = add_audit_cols(df_entregas_raw, "logistica_entregas.json", "Logística")

(
    df_entregas_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", True)
    .saveAsTable(f"{schema_name}.bronze_entregas")
)

print(df_entregas_bronze.count(), "linhas")
display(df_entregas_bronze.limit(5))

#### Ocorrências (NDJSON)

Schema variável entre registros: os campos `metadata` e `customer_code`
estão presentes em apenas parte das linhas. O leitor JSON do Spark trata
essa variação automaticamente, preenchendo `null` nos campos ausentes;
`mergeSchema` é necessário para a consolidação do schema na escrita.

In [ ]:
df_ocorrencias_raw = spark.read.json(f"{RAW_PATH}/atendimento_ocorrencias.ndjson")
df_ocorrencias_bronze = add_audit_cols(df_ocorrencias_raw, "atendimento_ocorrencias.ndjson", "Atendimento")

(
    df_ocorrencias_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", True)
    .saveAsTable(f"{schema_name}.bronze_ocorrencias")
)

print(df_ocorrencias_bronze.count(), "linhas")
df_ocorrencias_bronze.printSchema()

#### Resumo

In [ ]:
tabelas = [
    "bronze_pedidos_cabecalho", "bronze_pedidos_itens", "bronze_clientes",
    "bronze_produtos", "bronze_canais", "bronze_vendedores",
    "bronze_regioes", "bronze_entregas", "bronze_ocorrencias",
]

resumo = [(t, spark.table(f"{schema_name}.{t}").count()) for t in tabelas]
display(spark.createDataFrame(resumo, ["tabela", "linhas"]))